# 07 · Supporting baseline: PHM 2022 rock drill (sensor only)

Epiroc hydraulic rock drill in a **test cell**: eleven current conditions, with drill configurations held out; compares all 123 pressure features with 32 features selected on training configurations only. It is a different fault target from the Drill Rig gear-wear mapping. **Reproducibility note.** The committed result was produced with package versions that were not recorded. Under scikit-learn 1.9.1 this baseline gives different numbers (UCI hydraulic 0.963 vs committed 0.970, because its `StratifiedGroupKFold(shuffle=True)` split depends on the scikit-learn version and matches exactly under 1.6.1; ESPset 0.914 vs 0.918; PHM rock drill 0.798 vs 0.776, which stays different under either version and is not yet explained). This is a sensor-only benchmark and does not count as a sensor-versus-image experiment.

**Script:** `Models/rock_drill_baseline.py`  
**Result file:** `Results/phm2022_rock_drill.json`

In [1]:
REQUIRED = ['Raw_Data/phm2022_rock_drill/training_data.zip']
SCRIPT = 'Models/rock_drill_baseline.py'
JSON = 'phm2022_rock_drill.json'
ARGS = []
import contextlib
import io
import json
import os
import runpy
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
ROOT = next(p for p in (here, *here.parents) if (p / "Models").is_dir() and (p / "Preprocessing").is_dir())
RAW, PROCESSED, RESULTS = ROOT / "Raw_Data", ROOT / "Processed_Data", ROOT / "Results"

missing = [str(p.relative_to(ROOT)) for p in (ROOT / r for r in REQUIRED) if not p.exists()]
if missing:
    raise FileNotFoundError("Missing local data: " + ", ".join(missing) + ". Run Notebook/00_data_setup.ipynb first.")
for name in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"):
    os.environ[name] = "http://127.0.0.1:9"  # offline guard: any network call now fails loudly
os.environ["NO_PROXY"] = ""


def run_script(relative_path, *args, tail=3):
    """Run a project script exactly as `python <script>` would and show its last output lines."""
    path, buffer, saved = ROOT / relative_path, io.StringIO(), sys.argv
    sys.argv = [path.name, *args]
    try:
        with contextlib.redirect_stdout(buffer):
            runpy.run_path(str(path), run_name="__main__")
    finally:
        sys.argv = saved
    text = buffer.getvalue().replace(str(ROOT) + os.sep, "").replace(str(ROOT), ".")
    for line in [l for l in text.splitlines() if l.strip()][-tail:]:
        print("  " + line)


METRICS = ("accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1")


def metric_table(result):
    rows = {}
    def walk(node, path):
        if isinstance(node, dict):
            found = {k: round(node[k], 3) for k in METRICS if isinstance(node.get(k), (int, float))}
            if found:
                rows[" / ".join(path) or "(top level)"] = found
            for key, value in node.items():
                if isinstance(value, dict):
                    walk(value, [*path, key])
    walk(result, [])
    return pd.DataFrame(rows).T


def load(name):
    path = RESULTS / name
    if not path.is_file():
        return None
    text = path.read_text(encoding="utf-8")
    return json.loads(text) if name.endswith(".json") else text


print(f"Repository root: {ROOT.name}/ | data present | offline guard on")

Repository root: capstone-week4/ | data present | offline guard on


In [2]:
committed = load(JSON)
run_script(SCRIPT, *ARGS)
result = load(JSON)
if JSON.endswith(".json"):
    same = committed is not None and metric_table(committed).equals(metric_table(result))
else:
    same = committed == result
print(f"\n{JSON}: {'matches the committed result' if same else 'differs from the committed result (see note above)'}")
if JSON.endswith(".json"):
    display(metric_table(result))

  Inspected 34,045 cycles from configurations (1, 2, 4, 5, 6).
  Selected top_32_train_selected_features; held-out macro F1 0.798; accuracy 0.813.
  Wrote Results\phm2022_rock_drill.json and Figures\phm2022_rock_drill.png

phm2022_rock_drill.json: differs from the committed result (see note above)


,accuracy,balanced_accuracy,macro_f1
validation / full_features,0.703,0.705,0.657
validation / top_32_train_selected_features,0.790,0.791,0.767
test_selected,0.813,0.806,0.798
test_most_frequent,0.099,0.091,0.016


![Result figure](../Figures/phm2022_rock_drill.png)

*The figure shown is the committed version in `Figures/`; running this notebook regenerates it.*